# Layer Output Analysis

Characterize each layer's contribution: attn/MLP decomposition, prediction changes,
residual growth, information content, and uniqueness.

## Why This Matters

Layer output characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_output_analysis import (
    layer_output_decomposition, layer_prediction_change,
    layer_residual_growth, layer_information_content,
    layer_uniqueness_score,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Output Decomposition

Balance between attention and MLP at each layer.

In [ ]:
result = layer_output_decomposition(model, tokens)
print(f"Cooperative layers: {result['n_cooperative']}/4\n")
for p in result['per_layer']:
    coop = '+' if p['is_cooperative'] else '-'
    print(f"  Layer {p['layer']}: attn={p['attn_norm']:.4f}, mlp={p['mlp_norm']:.4f}, "
          f"combined={p['combined_norm']:.4f}, eff={p['efficiency']:.2%}, align={p['alignment']:+.4f} [{coop}]")

## Prediction Change

How does each layer change the model's prediction?

In [ ]:
result = layer_prediction_change(model, tokens)
print(f"Target: token {result['target_token']}\n")
for p in result['per_layer']:
    match = '✓' if p['matches_final'] else '✗'
    print(f"  Layer {p['layer']}: target_logit={p['target_logit']:+.4f}, "
          f"rank={p['target_rank']}, top={p['top_prediction']} [{match}]")

## Residual Growth

Magnitude and direction changes layer by layer.

In [ ]:
result = layer_residual_growth(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(min(p['relative_delta'], 1) * 20)
    print(f"  Layer {p['layer']}: {p['prev_norm']:.4f}→{p['curr_norm']:.4f}, "
          f"delta={p['delta_norm']:.4f} ({p['relative_delta']:.1%}), "
          f"align={p['delta_residual_alignment']:+.4f} {bar}")

## Information Content

Prediction entropy and confidence at each layer.

In [ ]:
result = layer_information_content(model, tokens)
sharp = 'SHARPENS' if result['sharpens_prediction'] else 'broadens'
print(f"Entropy reduction: {result['entropy_reduction']:.4f} [{sharp}]\n")
for p in result['per_layer']:
    bar = '█' * int(p['mean_confidence'] * 20)
    print(f"  Layer {p['layer']}: entropy={p['mean_entropy']:.4f}, "
          f"conf={p['mean_confidence']:.4f} {bar}")

## Uniqueness Score

Does each layer contribute something unique, or redundant?

In [ ]:
result = layer_uniqueness_score(model, tokens)
print(f"Unique layers: {result['n_unique']}/4\n")
for p in result['per_layer']:
    uniq = ' [UNIQUE]' if p['is_unique'] else ''
    bar = '█' * int(p['uniqueness_score'] * 20)
    print(f"  Layer {p['layer']}: uniqueness={p['uniqueness_score']:.4f}, "
          f"mean_sim={p['mean_abs_similarity']:.4f}, "
          f"max_sim={p['max_abs_similarity']:.4f}{uniq} {bar}")